# S2: manually build the PolyAccel RFSoC4x2 overlay

PolyAccel receives four float32 coefficients in each DATA header. Its AXI-Lite registers hold status only. The three board cases are cubic, identity and constant, with 100 samples each. DATA and END headers each occupy six 32-bit words.

This notebook calls the same application DAG as `python -m examples.stream_inband.poly_board`.
Use the notebook on the development computer with Vitis/Vivado **2025.1**. Run cells
in order. Both AMD-tool switches default to `False`; enable each stage when ready.
The physical board test is a later step using the portable PYNQ host.

The notebook writes a fresh run under `examples/stream_inband/board_build/notebook/`.
The independent command-line build writes under `examples/stream_inband/board_build/`.
Existing overlays and acceptance records are not selected or reused by this notebook.

## 1. Select the notebook environment

In a terminal at the **new `board_packaging` checkout root**, register its environment:

```bash
source .venv/bin/activate
unset PYTHONPATH
# Install ipykernel first if this environment does not already provide it.
python -m ipykernel install --user --name waveflow-board-packaging \
  --display-name "Waveflow board_packaging"
```

Select **Waveflow board_packaging** in Jupyter or VS Code and restart the kernel.
If `ipykernel` is missing, install it with `python -m pip install ipykernel` in this
activated environment. The setup cell below prints the actual checkout, branch and
interpreter. Confirm that it points to the migrated checkout.

The kernel sets its own `WAVEFLOW_VITIS_PATH` and `WAVEFLOW_VIVADO_PATH` because a
Jupyter/VS Code kernel may not inherit a terminal's exports. Edit these variables
before the tool stages if the installation is elsewhere. This setup does not run
tool executables.

In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys
import tempfile

# Use the nearest checkout when Jupyter starts beside this notebook.
repo = next((path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (path / ".git").exists() and (path / "waveflow/board/spec.py").is_file()), None)
if repo is None:
    raise RuntimeError("Open this notebook from the checkout containing board packaging.")
os.environ.pop("PYTHONPATH", None)
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "waveflow-mpl"))
tools_root = Path.home() / "Xilinx" / "2025.1"
os.environ.setdefault("WAVEFLOW_VITIS_PATH", str(tools_root / "Vitis/bin/vitis-run"))
os.environ.setdefault("WAVEFLOW_VIVADO_PATH", str(tools_root / "Vivado/bin/vivado"))
sys.path.insert(0, str(repo))
import waveflow
if not Path(waveflow.__file__).resolve().is_relative_to(repo):
    raise RuntimeError("Waveflow was imported from another checkout. Select the new kernel and restart it.")
branch = subprocess.run(["git", "branch", "--show-current"], cwd=repo,
                        check=True, capture_output=True, text=True).stdout.strip()
print("Checkout:", repo)
print("Branch:", branch)
print("Python:", sys.executable)
print("Waveflow:", waveflow.__file__)
for name in ("WAVEFLOW_VITIS_PATH", "WAVEFLOW_VIVADO_PATH"):
    print(name, "=", os.environ[name], "(exists:", Path(os.environ[name]).is_file(), ")")

## 2. Configure the board and build directory

`board_config` constructs the existing `BuildConfig` with the RFSoC4x2 board,
`ps_control` profile and optional calibration platform. Both the PS PL clock
request and HLS target are 100 MHz. A platform identity is created or confirmed;
it does not by itself contain calibrated timing/resource measurements.

Re-running this cell creates a new notebook run directory. Set
`WAVEFLOW_NOTEBOOK_BUILD_ROOT` before this cell to choose another output parent.
Use `WAVEFLOW_NOTEBOOK_PLATFORMS_ROOT` to override the shared platform library.

In [ ]:
from examples.stream_inband.poly_board import board_config, build_poly_board_dag
from waveflow.build.build import BuildDag, SourceStep
from waveflow.build.board_steps import BitstreamStep, CollectOverlayStep

example_dir = repo / "examples/stream_inband"
build_parent = Path(os.environ.get("WAVEFLOW_NOTEBOOK_BUILD_ROOT",
                                  str(example_dir / "board_build/notebook"))).resolve()
build_parent.mkdir(parents=True, exist_ok=True)
run_dir = Path(tempfile.mkdtemp(prefix="run-", dir=build_parent)).resolve()
platforms_root = Path(os.environ.get("WAVEFLOW_NOTEBOOK_PLATFORMS_ROOT",
                                    str(example_dir / "board_build/platforms"))).resolve()
config = board_config(run_dir, platform="rfsoc4x2_ps_control_100mhz",
                      platforms_root=platforms_root)
dag = build_poly_board_dag()
print("Notebook build:", run_dir)
print("Platform identity:", config.platform_info.manifest_path)
print("Board / part:", config.board.name, config.part)
print("PS clock / HLS target (Hz):", config.board.clock("ps_control").physical_hz, config.clk_freq)
print("Steps:", dag.step_names())

## 3. Check the Python model and host protocol

This cell runs through `host_sim` and reports failures immediately. It performs
software checks without invoking Vitis/Vivado or programming the board. The
model-only layout is distinguished from the actual ABI obtained during IP export.

The current upstream model and pysim scenarios are checked at both stream widths. The board host then checks cubic, identity and constant cases over the 32-bit DMA protocol.

In [ ]:
def run_checked(build_dag, through):
    def report(step, result):
        print(f"{step.name}: {'PASS' if result.success else 'FAIL'} {result.message}", flush=True)
    results = build_dag.run(config, through=through, force=True, on_step_end=report)
    failed = {name: result.traceback or result.message
              for name, result in results.items() if not result.success}
    if through not in results or failed:
        raise RuntimeError(f"Build did not complete {through}: {failed}")
    return results

local_results = run_checked(dag, "host_sim")
local_record = json.loads(local_results["host_sim"].path("host_sim_result").read_text())
assert local_record["passed"] and local_record["board_execution_verified"] is False
for name, result in local_record["cases"].items():
    print(name, {key: result[key] for key in ("passed", "sample_count", "max_abs_error")})

## 4. Manually run HLS, export, driver checks and BD validation

Set `RUN_BD_VALIDATION = True` in the next cell and execute it when ready.
It generates C++, synthesizes for RFSoC4x2, checks C/RTL results, exports IP,
validates the control ABI, generates and exercises the actual register driver,
and calls Vivado BD validation with the committed template.

The tool runners enforce version 2025.1. This stage ends before Vivado synthesis
and implementation. Keep this kernel session open to use its exact validation
and driver receipts in the next stage.

In [ ]:
RUN_BD_VALIDATION = False  # Set True, then run this cell manually.
validated = None
if RUN_BD_VALIDATION:
    missing = [name for name in ("WAVEFLOW_VITIS_PATH", "WAVEFLOW_VIVADO_PATH")
               if not Path(os.environ[name]).is_file()]
    if missing:
        raise FileNotFoundError(f"Set the installed tool paths first: {missing}")
    validated = run_checked(dag, "bd_validate")
    verification = json.loads(validated["synthesize"].path("hls_verification").read_text())
    bd_receipt = validated["bd_validate"].path("bd_validation")
    bd_record = json.loads(bd_receipt.read_text())
    assert verification["passed"]
    print("Fresh HLS verification:", verification)
    print("BD validation:", bd_receipt)
    print("Clock readback (Hz):", bd_record["verdict"]["physical_hz"])
    print("Address map:", bd_record["verdict"]["addresses"])
    print("Generated driver manifest:", validated["host_driver"].path("host_driver_manifest"))
else:
    print("AMD tools are disabled. Set RUN_BD_VALIDATION = True to run this stage.")

## 5. Manually implement and collect the overlay

After BD validation passes, set `RUN_IMPLEMENTATION = True` and run the next cell.
The continuation DAG uses the exact validation and driver receipts above, so it
does not repeat HLS synthesis/export. It runs Vivado synthesis/implementation,
checks timing and DRC, and collects BIT/HWH, driver.py and one manifest into a unique
overlay directory. XSA, reports and rebuild inputs remain in the build directory.

The notebook prints the precise manifest from this run. A collected overlay is
not yet a physical-board acceptance result.

In [ ]:
RUN_IMPLEMENTATION = False  # Set True after a successful BD validation.
overlay_manifest = None
implemented = None
if RUN_IMPLEMENTATION:
    if validated is None:
        raise RuntimeError("Run the BD validation cell successfully before implementation.")
    continuation = BuildDag()
    continuation.add(SourceStep(artifact="bd_validation",
                                path=validated["bd_validate"].path("bd_validation")))
    continuation.add(SourceStep(artifact="host_driver_manifest",
                                path=validated["host_driver"].path("host_driver_manifest")))
    continuation.add(BitstreamStep(name="bitstream", output_dir="implementation"))
    continuation.add(CollectOverlayStep(name="overlay", output_dir="overlays"))
    implemented = run_checked(continuation, "overlay")
    overlay_manifest = implemented["overlay"].path("overlay_manifest")
    print("Notebook overlay manifest:", overlay_manifest)
else:
    print("Implementation is disabled. Set RUN_IMPLEMENTATION = True after BD validation.")

## 6. Inspect this overlay and check its collected driver

The following cell uses only the overlay returned by this notebook run. After a
build it checks that overlay's portable driver against the current Python model.
When the tool stages are disabled it leaves the overlay unset.

In [ ]:
if overlay_manifest is None:
    print("No overlay was built by this notebook run.")
else:
    overlay_record = json.loads(overlay_manifest.read_text())
    print("Manifest:", overlay_manifest)
    print("Target:", overlay_record["target"])
    print("ABI SHA-256:", overlay_record["abi_sha256"])

    from examples.stream_inband.board.host import run_poly as host
    from examples.stream_inband.board.host.sim_backend import SimBackend
    verified_manifest, verified_record = host.verify_overlay(overlay_manifest)
    module = host.load_driver(verified_manifest.parent / "driver.py")
    assert module.ABI_SHA256 == verified_record["abi_sha256"]
    for name in ("cubic", "identity", "constant"):
        transport = SimBackend(module.ABI)
        try:
            result = host.run_transaction(module.Driver(transport), transport,
                host.load_vectors(run_dir / "vectors" / f"{name}.json"), poll_interval=0)
            assert result["passed"]
            print(name, "collected-driver model check:", result["sample_count"], "samples passed")
        finally:
            transport.close()
    print("Physical board execution verified: False (run the portable host on RFSoC4x2 next)")

## 7. Later: run fresh acceptance on RFSoC4x2

Copy the entire collected overlay directory as `overlay/`, and copy `examples/stream_inband/board/host/run_poly.py` and `dma.py` as `host/` files. Also copy this run's `vectors/` directory, including `cubic.json`, `identity.json` and `constant.json`.
Use a fresh test directory on the board. The following commands run **on the board**
from that directory, using its PYNQ-enabled Python environment:

```bash
for case in cubic identity constant; do
  python host/run_poly.py --backend pynq --overlay overlay/manifest.json \
    --vectors "vectors/$case.json" --output "board-acceptance-s2-$case.json" || break
done
```

Return the new acceptance JSON files after testing. The development-computer
simulation and build receipts do not replace this physical-board check. Choose
new output filenames for another run; the host refuses to overwrite records.

## 8. CMD: independently build a second overlay (with environment)

Run these commands manually in a Bash terminal at the **new checkout root**.
They use the same application DAG and current upstream CLI. This is an independent
full build in the command-line output root, separate from this notebook's run.

```bash
source .venv/bin/activate
unset PYTHONPATH
export WAVEFLOW_VITIS_PATH="$HOME/Xilinx/2025.1/Vitis/bin/vitis-run"
export WAVEFLOW_VIVADO_PATH="$HOME/Xilinx/2025.1/Vivado/bin/vivado"
export MPLCONFIGDIR=/tmp/waveflow-mpl

python -m examples.stream_inband.poly_board \
  --platform rfsoc4x2_ps_control_100mhz \
  --platforms-root examples/stream_inband/board_build/platforms \
  --through overlay --force --timing
```

For an initial tool validation without implementation, change `--through overlay`
to `--through bd_validate`. For a software-only run use `--through host_sim`.
A later `--force` command repeats the selected stage's prerequisites.

The CMD overlay manifests are under
`examples/stream_inband/board_build/overlays/overlay-*/manifest.json`.
Select the manifest from the new successful CMD run for its separate board test.
Its fresh vectors are under examples/stream_inband/board_build/vectors/. Copy those vectors with the CMD overlay for its board test.